#### Feature Engineering – Adult Income Dataset



Feature engineering is a critical step in the machine learning pipeline. It involves transforming raw data into meaningful features that improve model performance. For the Adult Income dataset, feature engineering focuses on: 
1.Creating new features
2.Transforming skewed features
3.Encoding categorical variables

In [22]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [23]:
df =pd.read_csv(r"C:\Users\gowri\OneDrive\MLAssignment\df_train_test.csv")

In [24]:
df.info()



<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48813 entries, 0 to 48812
Data columns (total 20 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   age               48813 non-null  int64 
 1   workclass         48813 non-null  object
 2   fnlwgt            48813 non-null  int64 
 3   education         48813 non-null  object
 4   education_num     48813 non-null  int64 
 5   marital_status    48813 non-null  object
 6   occupation        48813 non-null  object
 7   relationship      48813 non-null  object
 8   race              48813 non-null  object
 9   sex               48813 non-null  object
 10  capital_gain      48813 non-null  int64 
 11  capital_loss      48813 non-null  int64 
 12  hours_per_week    48813 non-null  int64 
 13  native_country    48813 non-null  object
 14  income            48813 non-null  object
 15  source            48813 non-null  object
 16  capital_gain_bin  48813 non-null  int64 
 17  capital_loss

In [25]:
print("Dataset shape:", df.shape)
df.head()

Dataset shape: (48813, 20)


,age,workclass,fnlwgt,education,education_num,marital_status,occupation,relationship,race,sex,capital_gain,capital_loss,hours_per_week,native_country,income,source,capital_gain_bin,capital_loss_bin,net_capital,income_binary
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K,train,1,0,2174,0
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K,train,0,0,0,0
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K,train,0,0,0,0
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K,train,0,0,0,0
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K,train,0,0,0,0


In [26]:
#log transformation for skewed features
skewed_features = ["capital_gain", "capital_loss"]

for col in skewed_features:
    df["log_" + col] = np.log1p(df[col])

In [27]:
#dropping the fnlwgt column
df = df.drop(columns=['fnlwgt'])

### Feature creation

In [13]:
#creating age groups
bins = [0, 25, 45, 65, 100]
labels = ["Young", "Adult", "Middle_Age", "Senior"]

df["age_group"] = pd.cut(df["age"], bins=bins, labels=labels)

In [28]:
#work hours category
df["hours_category"] = pd.cut(
    df["hours_per_week"],
    bins=[0, 25, 40, 60, 100],
    labels=["Part-time", "Full-time", "Overtime", "Heavy"]
)

In [29]:
#capital balance feature
df["capital_balance"] = df["net_capital"].apply(
    lambda x: "Loss" if x < 0 else ("Gain" if x > 0 else "None")
)

In [30]:
#education level grouping
education_map = {
    "Preschool": "Low",
    "1st-4th": "Low",
    "5th-6th": "Low",
    "7th-8th": "Low",
    "9th": "Low",
    "10th": "Medium",
    "11th": "Medium",
    "12th": "Medium",
    "HS-grad": "Medium",
    "Some-college": "High",
    "Assoc-acdm": "High",
    "Assoc-voc": "High",
    "Bachelors": "High",
    "Masters": "Very_High",
    "Prof-school": "Very_High",
    "Doctorate": "Very_High"
}

df["education_level"] = df["education"].map(education_map)

In [31]:
#occupation skill grouping
high_skill = [
    "Exec-managerial","Prof-specialty"
]

medium_skill = [
    "Tech-support","Sales","Adm-clerical"
]

low_skill = [
    "Craft-repair","Machine-op-inspct","Transport-moving",
    "Handlers-cleaners","Farming-fishing","Priv-house-serv"
]

def occupation_group(x):
    if x in high_skill:
        return "High_skill"
    elif x in medium_skill:
        return "Medium_skill"
    elif x in low_skill:
        return "Low_skill"
    else:
        return "Other"

df["occupation_skill"] = df["occupation"].apply(occupation_group)

In [32]:
#native country grouping
df["country_group"] = df["native_country"].apply(
    lambda x: "US" if x == "United-States" else "Non-US"
)

In [35]:
df.drop(["education","occupation","native_country"], axis=1, inplace=True)

In [37]:
#creating workclass group
workclass_map = {
    "Private": "Private",
    "Self-emp-not-inc": "Self_Employed",
    "Self-emp-inc": "Self_Employed",
    "Federal-gov": "Government",
    "Local-gov": "Government",
    "State-gov": "Government",
    "Without-pay": "Unemployed",
    "Never-worked": "Unemployed"
}

df["workclass_group"] = df["workclass"].map(workclass_map)

In [38]:
df.drop("workclass", axis=1, inplace=True)

In [39]:
#creating is_married feature
df["is_married"] = df["marital_status"].apply(
    lambda x: 1 if "Married" in x else 0
)

In [40]:
df.drop("marital_status", axis=1, inplace=True)

In [41]:
#relationship
df["family_role"] = df["relationship"].apply(
    lambda x: "Family" if x in ["Husband","Wife","Own-child"] else "Non-Family"
)

In [42]:
df.drop("relationship", axis=1, inplace=True)

In [43]:
#race
df["race_group"] = df["race"].apply(
    lambda x: "White" if x == "White" else "Non-White"
)

In [44]:
df.drop("race", axis=1, inplace=True)

In [45]:
#sex
df["sex_binary"] = df["sex"].map({
    "Male": 1,
    "Female": 0
})

In [46]:
df.drop("sex", axis=1, inplace=True)

In [47]:
categorical_cols = df.select_dtypes(include=["object", "category"]).columns.tolist()
numerical_cols = df.select_dtypes(include=["int64", "float64"]).columns.tolist()

print("Categorical Columns:", categorical_cols)
print("Numerical Columns:", numerical_cols)

Categorical Columns: ['income', 'source', 'hours_category', 'capital_balance', 'education_level', 'occupation_skill', 'country_group', 'workclass_group', 'family_role', 'race_group']
Numerical Columns: ['age', 'education_num', 'capital_gain', 'capital_loss', 'hours_per_week', 'capital_gain_bin', 'capital_loss_bin', 'net_capital', 'income_binary', 'log_capital_gain', 'log_capital_loss', 'is_married', 'sex_binary']


In [48]:
#removing target variable before encoding
categorical_cols.remove("income")

In [49]:
df_encoded = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

print("Encoded Dataset Shape:", df_encoded.shape)
df_encoded.head()

Encoded Dataset Shape: (48813, 32)


,age,education_num,capital_gain,capital_loss,hours_per_week,income,capital_gain_bin,capital_loss_bin,net_capital,income_binary,...,education_level_Very_High,occupation_skill_Low_skill,occupation_skill_Medium_skill,occupation_skill_Other,country_group_US,workclass_group_Private,workclass_group_Self_Employed,workclass_group_Unemployed,family_role_Non-Family,race_group_White
0,39,13,2174,0,40,<=50K,1,0,2174,0,...,False,False,True,False,True,False,False,False,True,True
1,50,13,0,0,13,<=50K,0,0,0,0,...,False,False,False,False,True,False,True,False,False,True
2,38,9,0,0,40,<=50K,0,0,0,0,...,False,True,False,False,True,True,False,False,True,True
3,53,7,0,0,40,<=50K,0,0,0,0,...,False,True,False,False,True,True,False,False,False,False
4,28,13,0,0,40,<=50K,0,0,0,0,...,False,False,False,False,False,True,False,False,False,False


In [50]:
#defining features and target
X = df_encoded.drop(["income_binary","income"], axis=1)
y = df_encoded["income_binary"]

print("Feature matrix shape:", X.shape)
print("Target shape:", y.shape)

Feature matrix shape: (48813, 30)
Target shape: (48813,)


In [52]:
df_encoded.head()

,age,education_num,capital_gain,capital_loss,hours_per_week,income,capital_gain_bin,capital_loss_bin,net_capital,income_binary,...,education_level_Very_High,occupation_skill_Low_skill,occupation_skill_Medium_skill,occupation_skill_Other,country_group_US,workclass_group_Private,workclass_group_Self_Employed,workclass_group_Unemployed,family_role_Non-Family,race_group_White
0,39,13,2174,0,40,<=50K,1,0,2174,0,...,False,False,True,False,True,False,False,False,True,True
1,50,13,0,0,13,<=50K,0,0,0,0,...,False,False,False,False,True,False,True,False,False,True
2,38,9,0,0,40,<=50K,0,0,0,0,...,False,True,False,False,True,True,False,False,True,True
3,53,7,0,0,40,<=50K,0,0,0,0,...,False,True,False,False,True,True,False,False,False,False
4,28,13,0,0,40,<=50K,0,0,0,0,...,False,False,False,False,False,True,False,False,False,False


In [53]:
df_encoded.to_csv("adult_feature_engineered.csv", index=False)